In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
from src.evaluation.explainer import PhishingExplainer
from src.features.extractor import PhishingFeatureExtractor

/Users/7312/Desktop/Repositories/phishing-transformer-detection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
explainer = PhishingExplainer("../saved_models/herbert-base")
extractor = PhishingFeatureExtractor()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2534.42it/s, Materializing param=classifier.weight]                                      


In [3]:
def run_phishing_audit(test_examples, explainer_model, extractor):
    results = []

    for i, item in enumerate(test_examples):
        text = item["text"]
        true_label = item["label"]

        # 1. Feature extraction:
        features = extractor.get_all_features(text)

        heuristic_score = sum(
            [
                features["urgency_score"],
                features["threat_score"],
                features["fin_score"],
                features["emo_score"],
                features["has_suspicious_tld"] * 3,
            ]
        )

        # 2. Model prediction:
        explanation = explainer_model.get_explanation(text)
        model_confidence = np.sum(explanation.values[0][:, 1])
        prediction = "PHISH" if model_confidence > 0 else "LEGIT"

        results.append(
            {
                "ID": i + 1,
                "True Label": true_label,
                "Model Decision": prediction,
                "Model Confidence (SHAP Sum)": round(model_confidence, 4),
                "Heuristic Score (Tags)": heuristic_score,
                "EMO": features["emo_score"],
                "FIN": features["fin_score"],
                "URL_Alert": features["has_suspicious_tld"] or features["has_homograph_attack"],
                "Content Snippet": text[:50] + "...",
            }
        )

    df_results = pd.DataFrame(results)

    def _highlight_discrepancy(row):
        color = ""

        if row["True Label"] != row["Model Decision"]:
            color = "background-color: #ffcccc"

        elif row["Model Confidence (SHAP Sum)"] < 0.2 and row["Model Decision"] == "PHISH":
            color = "background-color: #fff3cd"

        return [color] * len(row)

    return df_results.style.apply(_highlight_discrepancy, axis=1)

In [4]:
test_data = [
    {"label": "PHISH", "text": "[SENDER] Urząd Skarbowy [CONTENT] PILNE! Masz zaległość VAT 14.50 PLN. Opłać teraz!"},
    {"label": "PHISH", "text": "[SENDER] PKO BP [CONTENT] Twoje konto zostało zablokowane. https://secure-pko-bp.com"},
    {"label": "LEGIT", "text": "[SENDER] PKO BP [CONTENT] Potwierdzenie wykonania przelewu nr 12345."},
    {
        "label": "PHISH",
        "text": "[SENDER] Netflix [CONTENT] Twoja subskrypcja wygasła! Zaktualizuj płatność: https://netflix-update.xyz",
    },
]  # TODO: Modify these examples to better reflect the features we are extracting (e.g. add more urgency, financial terms, etc.)

In [5]:
audit_report = run_phishing_audit(test_data, explainer, extractor)
display(audit_report)

,ID,True Label,Model Decision,Model Confidence (SHAP Sum),Heuristic Score (Tags),EMO,FIN,URL_Alert,Content Snippet
0,1,PHISH,LEGIT,-0.529100,6,5,0,0,[SENDER] Urząd Skarbowy [CONTENT] PILNE! Masz zale...
1,2,PHISH,PHISH,0.416200,4,3,0,0,[SENDER] PKO BP [CONTENT] Twoje konto zostało zabl...
2,3,LEGIT,LEGIT,-0.513700,4,3,1,0,[SENDER] PKO BP [CONTENT] Potwierdzenie wykonania ...
3,4,PHISH,PHISH,0.463200,7,3,1,1,[SENDER] Netflix [CONTENT] Twoja subskrypcja wygas...
